# **Causal graphs: d-separation, backdoor, front-door and mediation**

Practice for the module [“Causality and Linear Models”](https://ai-interpretability.school), lesson “Graphs: d-separation, backdoor and front-door”.

In the lesson the graph was a picture with arrows, and the rules were formulas. Here we define the graph by structural equations, generate data from it and check four claims of the lesson on the data:

1. d-separation in the graph means conditional independence in the data, and the backdoor criterion chooses the adjustment set without guessing;
2. a variable that is both a confounder and a collider biases the estimate when adjusted for alone, although it “looks like the right control”;
3. the front-door formula recovers the effect when a common cause of the treatment and the outcome is not measured;
4. the total effect splits into a direct and an indirect one, and the split breaks when the mediator and the outcome have a hidden common cause.

Plan of the notebook:

1. the graph and data from structural equations;
2. d-separation and partial correlations;
3. the backdoor criterion: all valid adjustment sets;
4. the adjustment formula and effect estimates for different sets;
5. front-door on binary data with a hidden confounder;
6. direct and indirect effect;
7. where it breaks: a mediator and a collider among the controls, a hidden confounder of the mediator;
8. the same computations in DoWhy.

All data are synthetic: only then is the true effect known and an estimate has something to be compared with. The notebook runs on a CPU, the whole run takes about a minute.

In [ ]:
%pip install -q dowhy==0.14

In [ ]:
import itertools
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.mediation import Mediation

warnings.filterwarnings('ignore')
SEED = 0
pd.set_option('display.width', 140)

## 1. The graph and the data

A structural causal model (SCM) defines every variable by an equation of its parents in the graph and its own noise. The story is the one from the lesson: a professional training course $T$ (hours of training) and salary $Y$.

- $X_1$ is work experience, $X_2$ is the network of professional contacts; they do not depend on each other;
- $X_3$ is the manager's rating: it depends on both experience and contacts, and it affects how many training hours an employee gets and the salary;
- $M$ is the skill the course gives; $C$ is an invitation to an interview at another company, a consequence of both the course and the salary.

$$X_1=\varepsilon_1,\quad X_2=\varepsilon_2,\quad X_3=0.8X_1+0.8X_2+\varepsilon_3,\quad T=0.7X_1+0.6X_3+\varepsilon_T,$$
$$M=0.9T+\varepsilon_M,\quad Y=0.5T+0.8M+0.7X_2+0.6X_3+\varepsilon_Y,\quad C=0.6T+0.8Y+\varepsilon_C,$$

where all noises $\varepsilon$ are independent with distribution $\mathcal N(0,1)$.

The intervention $do(T=t)$ replaces the equation for $T$ by a constant and keeps the other equations as they are. Hence the true total effect of an hour of training is the sum of products of coefficients along the directed paths from $T$ to $Y$: $0.5 + 0.9\cdot 0.8 = 1.22$. The direct effect is $0.5$, the indirect one, through the skill, is $0.72$.

In [ ]:
G = nx.DiGraph([('X1', 'T'), ('X1', 'X3'), ('X2', 'X3'), ('X2', 'Y'), ('X3', 'T'), ('X3', 'Y'),
                ('T', 'M'), ('M', 'Y'), ('T', 'Y'), ('T', 'C'), ('Y', 'C')])

def simulate(n, seed=SEED):
    rng = np.random.default_rng(seed)
    e = lambda: rng.normal(size=n)
    X1 = e(); X2 = e()
    X3 = 0.8 * X1 + 0.8 * X2 + e()
    T = 0.7 * X1 + 0.6 * X3 + e()
    M = 0.9 * T + e()
    Y = 0.5 * T + 0.8 * M + 0.7 * X2 + 0.6 * X3 + e()
    C = 0.6 * T + 0.8 * Y + e()
    return pd.DataFrame(dict(X1=X1, X2=X2, X3=X3, T=T, M=M, Y=Y, C=C))

df = simulate(20000)
TRUE_TOTAL, TRUE_DIRECT, TRUE_INDIRECT = 0.5 + 0.9 * 0.8, 0.5, 0.9 * 0.8
print('objects:', len(df), '| true total effect:', round(TRUE_TOTAL, 2))
df.head().round(2)

## 2. d-separation and partial correlations

A path between two nodes is blocked by a set $S$ if it contains a chain $A\to B\to C$ or a fork $A\leftarrow B\to C$ with the middle node $B$ in $S$, or a collider $A\to B\leftarrow C$ such that neither $B$ nor its descendants are in $S$. Two nodes are d-separated by $S$ if all paths between them are blocked. In `networkx` this is the function `nx.is_d_separator(G, x, y, S)`.

In a linear model with Gaussian noise, conditional independence is zero partial correlation: the correlation of the residuals of the two variables after regressing each of them on the set $S$. Let us compare the verdict of the graph with the data for several pairs.

In [ ]:
def partial_corr(data, a, b, S):
    if not S:
        return np.corrcoef(data[a], data[b])[0, 1]
    Z = sm.add_constant(data[list(S)])
    ra = sm.OLS(data[a], Z).fit().resid
    rb = sm.OLS(data[b], Z).fit().resid
    return np.corrcoef(ra, rb)[0, 1]

checks = [('X1', 'X2', []), ('X1', 'X2', ['X3']), ('X1', 'Y', ['X3', 'T']), ('X1', 'Y', ['X2', 'X3', 'T']),
          ('T', 'X2', ['X3']), ('T', 'X2', ['X1', 'X3']), ('M', 'X3', ['T'])]
rows = [dict(pair=f'{a} and {b}', set=', '.join(S) or '—', d_separated=nx.is_d_separator(G, {a}, {b}, set(S)),
             partial_correlation=round(partial_corr(df, a, b, S), 3)) for a, b, S in checks]
pd.DataFrame(rows)

Where the graph says “d-separated”, the partial correlation is within sampling noise (hundredths on 20,000 objects); where it says “not separated”, it is clearly different from zero. The second row is the collider $X_3$: experience and contacts are independent, but they become related as soon as we fix the manager's rating. With equal coefficients $0.8$ the relation is negative: at the same rating, an employee with more experience has on average fewer contacts.

## 3. The backdoor criterion

A set $S$ satisfies the backdoor criterion relative to the pair $(T, Y)$ if

1. $S$ contains no descendants of $T$;
2. $S$ blocks every path from $T$ to $Y$ that starts with an arrow into $T$.

The second condition is checked by d-separation in the graph with all arrows out of $T$ removed: only the “back doors” remain in it. Let us go through all subsets of the five measured variables $X_1, X_2, X_3, M, C$.

In [ ]:
def satisfies_backdoor(G, t, y, S):
    if set(S) & nx.descendants(G, t):
        return False
    G_back = G.copy()
    G_back.remove_edges_from(list(G.out_edges(t)))
    return nx.is_d_separator(G_back, {t}, {y}, set(S))

candidates = ['X1', 'X2', 'X3', 'M', 'C']
subsets = [S for k in range(len(candidates) + 1) for S in itertools.combinations(candidates, k)]
valid = [S for S in subsets if satisfies_backdoor(G, 'T', 'Y', S)]
print('subsets in total:', len(subsets))
print('valid sets:', len(valid), valid)
print('minimal set (networkx):', nx.find_minimal_d_separator(
    nx.restricted_view(G, [], list(G.out_edges('T'))), 'T', 'Y', restricted={'X1', 'X2', 'X3'}))

**Task 1. How many of the 32 subsets of $\{X_1, X_2, X_3, M, C\}$ satisfy the backdoor criterion? Why is the set with $X_3$ alone not among them, although $X_3$ is a common cause of $T$ and $Y$?**

## 4. The adjustment formula

If a set $S$ satisfies the backdoor criterion, then

$$P(y\mid do(t))=\sum_s P(y\mid t,s)\,P(s).$$

In a linear model without interactions the sum collapses: the effect of a unit of $T$ is the coefficient of $T$ in the regression of $Y$ on $T$ and $S$. Let us compute it for the empty set, for every valid set, for $X_3$ alone and for sets with the mediator and the collider.

In [ ]:
def adjusted_effect(data, S):
    formula = 'Y ~ T' + ''.join(' + ' + s for s in S)
    return smf.ols(formula, data=data).fit().params['T']

sets = [(), ('X3',)] + valid + [('X2', 'X3', 'M'), ('X2', 'X3', 'C')]
table = pd.DataFrame([dict(set=', '.join(S) or '—', backdoor=satisfies_backdoor(G, 'T', 'Y', S),
                           estimate=round(adjusted_effect(df, S), 3)) for S in sets])
table['error'] = (table['estimate'] - TRUE_TOTAL).round(3)
table

**Task 2. What is the effect estimate for the set $\{X_2, X_3\}$ (rounded to three decimal places)? How far is it from the estimate with $X_3$ alone, and in which direction is the estimate without adjustment biased?**

All three valid sets give estimates within noise of the true $1.22$. $X_3$ alone closes the forks $T\leftarrow X_3\to Y$ and $T\leftarrow X_1\to X_3\to Y$, but opens the path $T\leftarrow X_1\to X_3\leftarrow X_2\to Y$, where $X_3$ is a collider. The mediator among the controls cuts off the indirect effect, and the direct one, about $0.5$, remains; the collider among the controls biases the estimate most of all.

## 5. Front-door: a hidden confounder

A new graph: a drive for self-improvement $U$ (not in the data) affects both whether an employee takes the course $T$ and her promotion $Y$; the course gives a certificate $M$, and the promotion depends on the certificate, not on the course itself. All variables are binary:

$$U\sim\text{Bern}(0.5),\quad T\sim\text{Bern}(0.2+0.6U),\quad M\sim\text{Bern}(0.1+0.7T),\quad Y\sim\text{Bern}(0.1+0.5M+0.3U).$$

The true effect: $P(Y=1\mid do(T=1))-P(Y=1\mid do(T=0)) = 0.5\cdot 0.7 = 0.35$. Backdoor does not apply here: the only “back door” $T\leftarrow U\to Y$ goes through the unmeasured $U$. But $M$ satisfies the front-door criterion, and

$$P(y\mid do(t))=\sum_m P(m\mid t)\sum_{t'}P(y\mid m,t')\,P(t').$$

In [ ]:
rng = np.random.default_rng(SEED)
n_fd = 20000
U = rng.binomial(1, 0.5, n_fd)
T_fd = rng.binomial(1, 0.2 + 0.6 * U)
M_fd = rng.binomial(1, 0.1 + 0.7 * T_fd)
Y_fd = rng.binomial(1, 0.1 + 0.5 * M_fd + 0.3 * U)
fd = pd.DataFrame(dict(T=T_fd, M=M_fd, Y=Y_fd))   # U does not enter the table
TRUE_FD = 0.5 * 0.7

naive_fd = fd.loc[fd['T'] == 1, 'Y'].mean() - fd.loc[fd['T'] == 0, 'Y'].mean()
print('naive difference of shares:', round(naive_fd, 3), '| true effect:', TRUE_FD)

In [ ]:
p_t = fd['T'].value_counts(normalize=True).sort_index()   # P(t')
p_m1_t = fd.groupby('T')['M'].mean()                      # P(M=1 | t)
p_y1_mt = fd.groupby(['M', 'T'])['Y'].mean()              # P(Y=1 | m, t')

def p_y1_do(t):
    total = 0.0
    for m in (0, 1):
        p_m = p_m1_t[t] if m == 1 else 1 - p_m1_t[t]
        total += p_m * sum(p_y1_mt[(m, tp)] * p_t[tp] for tp in (0, 1))
    return total

frontdoor = p_y1_do(1) - p_y1_do(0)
print('P(Y=1 | do(T=1)) =', round(p_y1_do(1), 3), '| P(Y=1 | do(T=0)) =', round(p_y1_do(0), 3))
print('front-door estimate:', round(frontdoor, 3))

**Task 3. What is the front-door estimate of the course effect (rounded to three decimal places)? How many times is the naive difference of shares larger than the true effect $0.35$?**

The inner sum over $t'$ is the same adjustment formula, only for the pair $(M, Y)$: the path $M\leftarrow T\leftarrow U\to Y$ is closed by adjusting for $T$. The outer sum carries over the effect of $T$ on $M$, which is unconfounded because $T$ and $M$ have no common causes.

## 6. Direct and indirect effect

Back to the first graph. In a linear model without a $T\times M$ interaction, the indirect effect is the product of coefficients $a\cdot b$ ([Baron and Kenny, 1986](https://doi.org/10.1037/0022-3514.51.6.1173)): $a$ is the coefficient of $T$ in the regression of $M$ on $T$ and the controls, $b$ is the coefficient of $M$ in the regression of $Y$ on $T$, $M$ and the controls. The direct effect is the coefficient of $T$ in the second regression, and together they give the total effect. The controls are the valid set $\{X_2, X_3\}$: it closes both the “back doors” of $T$ and the common causes of $M$ and $Y$ other than $T$.

`statsmodels.stats.mediation.Mediation` computes the same by simulation, following the algorithm of [Imai, Keele and Tingley (2010)](https://doi.org/10.1037/a0020761): ACME is the average indirect effect, ADE the average direct one.

In [ ]:
def product_of_coefficients(data, controls):
    c = ''.join(' + ' + s for s in controls)
    a = smf.ols('M ~ T' + c, data=data).fit().params['T']
    out = smf.ols('Y ~ T + M' + c, data=data).fit()
    return a, out.params['M'], out.params['T']

a, b, direct = product_of_coefficients(df, ['X2', 'X3'])
print(f'a = {a:.3f}, b = {b:.3f}')
print(f'indirect a*b = {a * b:.3f} (true {TRUE_INDIRECT:.2f}), direct = {direct:.3f} (true {TRUE_DIRECT})')
print(f'sum = {a * b + direct:.3f}, share of indirect = {a * b / (a * b + direct):.3f}')

med = Mediation(sm.OLS.from_formula('Y ~ T + M + X2 + X3', df), sm.OLS.from_formula('M ~ T + X2 + X3', df),
                'T', 'M').fit(n_rep=200, rng=np.random.default_rng(SEED))
med.summary().loc[['ACME (average)', 'ADE (average)', 'Total effect', 'Prop. mediated (average)']].round(3)

**Task 4. What is the indirect effect $a\cdot b$ with the controls $\{X_2, X_3\}$ (rounded to three decimal places)? What share of the total effect goes through the skill?**

## 7. Where it breaks

### 7.1. A mediator and a collider among the controls

The table of section 4 already showed both cases: with the mediator among the controls the regression estimates the direct effect instead of the total one, with the collider it estimates a quantity equal to neither. The quality metric of the regression grows at the same time: $C$ is a consequence of $Y$ and predicts it well.

In [ ]:
for S in [('X2', 'X3'), ('X2', 'X3', 'M'), ('X2', 'X3', 'C')]:
    fit = smf.ols('Y ~ T' + ''.join(' + ' + s for s in S), data=df).fit()
    print(f'{", ".join(S):12s} estimate {fit.params["T"]:.3f}   R^2 {fit.rsquared:.3f}')

### 7.2. A hidden common cause of the mediator and the outcome

The split into direct and indirect effects requires more than the total effect: besides the “back doors” of $T$, there must be no hidden common causes of $M$ and $Y$ (Imai and coauthors call this assumption sequential ignorability). Let us add to the first graph a motivation $V$ that raises both the skill and the salary, and not record it in the data. The total effect does not change: $V$ does not affect $T$.

In [ ]:
def simulate_hidden_mediator_confounder(n, strength, seed=SEED):
    rng = np.random.default_rng(seed)
    e = lambda: rng.normal(size=n)
    X1 = e(); X2 = e(); V = e()
    X3 = 0.8 * X1 + 0.8 * X2 + e()
    T = 0.7 * X1 + 0.6 * X3 + e()
    M = 0.9 * T + strength * V + e()
    Y = 0.5 * T + 0.8 * M + 0.7 * X2 + 0.6 * X3 + strength * V + e()
    return pd.DataFrame(dict(X1=X1, X2=X2, X3=X3, T=T, M=M, Y=Y))

rows = []
for strength in [0.0, 0.5, 1.0]:
    d = simulate_hidden_mediator_confounder(20000, strength)
    a_v, b_v, direct_v = product_of_coefficients(d, ['X2', 'X3'])
    rows.append(dict(strength_V=strength, total=round(adjusted_effect(d, ('X2', 'X3')), 3),
                     indirect_ab=round(a_v * b_v, 3), direct=round(direct_v, 3)))
pd.DataFrame(rows)

The total effect stays around $1.22$ for any strength of $V$, while the split drifts: $b$ absorbs the relation of $M$ and $Y$ through $V$, the indirect effect grows, the direct one goes down and almost vanishes at strength $1$. The data do not show it: the regressions are the same as in section 6. This is why Imai, Keele and Tingley propose to report a sensitivity analysis to the violation of this assumption together with the estimate.

## 8. The same computations in DoWhy

[DoWhy](https://github.com/py-why/dowhy) (MIT license) splits the work into the same steps: graph → identification (which criterion gives the formula) → estimation. `identify_effect` finds a set by the backdoor criterion by itself, and for the second graph the front-door formula; it treats the variable `U` in the graph as unmeasured because it is not in the table.

In [ ]:
from dowhy import CausalModel

model = CausalModel(data=df, treatment='T', outcome='Y', graph=G)
estimand = model.identify_effect()
print('backdoor set:', estimand.get_backdoor_variables())
est = model.estimate_effect(estimand, method_name='backdoor.linear_regression')
print('DoWhy, backdoor:', round(est.value, 3))

G_fd = nx.DiGraph([('U', 'T'), ('U', 'Y'), ('T', 'M'), ('M', 'Y')])
model_fd = CausalModel(data=fd, treatment='T', outcome='Y', graph=G_fd)
estimand_fd = model_fd.identify_effect()
print('front-door through:', estimand_fd.get_frontdoor_variables())
est_fd = model_fd.estimate_effect(estimand_fd, method_name='frontdoor.two_stage_regression')
print('DoWhy, front-door:', round(est_fd.value, 3), '| our formula:', round(frontdoor, 3))

`frontdoor.two_stage_regression` in DoWhy is the linear version of the formula: the coefficient of $T$ in the regression of $M$ on $T$, times the coefficient of $M$ in the regression of $Y$ on $M$ and $T$. On binary data this is an approximation, and it can differ from the exact sum of section 5 in the last digits.

## What to take away

- **The graph decides what to adjust for before any data.** The backdoor criterion is checked by d-separation in the graph without the arrows out of $T$; going through the subsets replaces guesses like “adjust for everything related to $T$”.
- **A confounder can also be a collider.** $X_3$ is a common cause of $T$ and $Y$, but adjusting for it alone opens a path through $X_1$ and $X_2$; only a set that also closes this path helps.
- **Front-door does without measuring the confounder,** but it requires the whole effect to go through a measured mediator and no hidden common causes of $T$ and $M$, nor of $M$ and $Y$ (apart from $T$ itself).
- **Direct and indirect effects require more assumptions than the total effect.** A hidden common cause of the mediator and the outcome does not touch the total effect but distorts the split, and the data do not show it.

Sources:
1. [Pearl, 1995. Causal diagrams for empirical research](https://doi.org/10.1093/biomet/82.4.702)
2. [Pearl, 2001. Direct and Indirect Effects](https://arxiv.org/abs/1301.2300)
3. [Baron & Kenny, 1986. The moderator–mediator variable distinction in social psychological research](https://doi.org/10.1037/0022-3514.51.6.1173)
4. [Imai, Keele & Tingley, 2010. A general approach to causal mediation analysis](https://doi.org/10.1037/a0020761)
5. [Sharma & Kiciman, 2020. DoWhy: An End-to-End Library for Causal Inference](https://arxiv.org/abs/2011.04216)